# Third Harmonic Generation

In this example, we consider wave propagation through a simple 1d nonlinear medium with a non-zero Kerr susceptibility $\chi^{(3)}$. See also [Materials](https://meep.readthedocs.io/en/latest/Materials/#nonlinearity) and [Units and Nonlinearity](https://meep.readthedocs.io/en/latest/Units_and_Nonlinearity/). We send in a narrow-band pulse at a frequency $\omega$, and because of the nonlinearity we also get a signal at a frequency $3\omega$.

Since this is a 1d calculation, we could implement it via a 2d cell of `Vector3(S,0,0)`, specifying periodic boundary conditions in the $y$ direction. However, this is slightly inefficient since the $y$ periodic boundaries are implemented internally via extra "ghost pixels" in the $y$ direction. Instead, Meep has special support for 1d simulations in the $z$ direction. To use this, we must explicitly set `dimensions` to 1 in the `Simulation` object constructor, and in that case we can only use $E_x$ (and $D_x$) and $H_y$ field components. This involves no loss of generality because of the symmetry of the problem.

We are going to fill the entire computational cell with the nonlinear medium, so we don't need to use any objects. We can just use the special `default_material` which is ordinarily vacuum. PMLs are placed at the $\pm 1$ boundaries. The source will be a Gaussian pulse of $J_x$ just next to the $-z$ PML layer. Since this is a nonlinear calculation, we may want to play with the amplitude of the current/field, so we set the `amplitude` property explicitly to a parameter `amp`.

In [ ]:
from typing import Tuple, List, Union
import numpy as np
from matplotlib import pyplot as plt
import meep as mp

In [ ]:
def third_harmonic_generation(
    k: float, amp: float = 1.0, nfreq: int = 10, flux_spectrum: bool = True
) -> Union[Tuple[List[float], List[float]], Tuple[float, float]]:
    
    """Computes the transmission spectrum of a plane wave propagating                                             
       through a Kerr medium.                                                                                     
                                                                                                                  
    Args:                                                                                                         
      k: strength of Kerr susceptibility.                                                                         
      amp: amplitude of the incident planewave.                                                                   
      nfreq: number of frequencies in flux spectrum.                                                              
      flux_spectrum: compute the flux spectrum over broad bandwidth (True) or                                     
                     just the two harmonic frequencies ω and 3ω (False).                                          
                                                                                                                  
    Returns:                                                                                                      
      The frequencies and transmitted flux over the flux spectrum or                                              
      the transmitted flux at the harmonic frequencies ω and 3ω.                                                  
    """
    
    sz = 100  # size of cell in z direction                                                                       
    fcen = 1 / 3.0  # center frequency of source                                                                  
    df = fcen / 20.0  # frequency width of source                                                                 
    dpml = 1.0  # PML thickness                                                                                   
                                                                   
    dimensions = 1
    cell = mp.Vector3(0, 0, sz)
    pml_layers = [mp.PML(dpml)]
    resolution = 25
                                                                
    default_material = mp.Medium(index=1, chi3=k)

    sources = [
        mp.Source(
            mp.GaussianSource(fcen, fwidth=df),
            component=mp.Ex,
            center=mp.Vector3(0, 0, -0.5 * sz + dpml),
            amplitude=amp,
        )
    ]

    # frequency range for flux calculation                                                                        
    fmin = fcen / 2.0
    fmax = fcen * 4

    sim = mp.Simulation(
        cell_size=cell,
        sources=sources,
        boundary_layers=pml_layers,
        default_material=default_material,
        resolution=resolution,
        dimensions=dimensions,
    )

    mon_pt = mp.Vector3(0, 0, 0.5 * sz - dpml - 0.5)
    
    if flux_spectrum:
        trans = sim.add_flux(
            0.5 * (fmin + fmax),
            fmax - fmin,
            nfreq,
            mp.FluxRegion(mon_pt),
        )
    else:
        trans1 = sim.add_flux(fcen, 0, 1, mp.FluxRegion(mon_pt))
        trans3 = sim.add_flux(
            3 * fcen, 0, 1, mp.FluxRegion(mon_pt)
        )

    sim.run(until_after_sources=mp.stop_when_fields_decayed(50, mp.Ex, mon_pt, 1e-6))

    if flux_spectrum:
        freqs = mp.get_flux_freqs(trans)
        trans_flux = mp.get_fluxes(trans)
        return freqs, trans_flux
    else:
        print(
            f"harmonics:, {k}, {amp}, {mp.get_fluxes(trans1)[0]}, "
            f"{mp.get_fluxes(trans3)[0]}"
        )
        return mp.get_fluxes(trans1)[0], mp.get_fluxes(trans3)[0]

Note DFT decimation is turned off by default whenever nonlinearities are present.

In the first part of this tutorial, we plot the transmitted power spectrum for various values of $\chi^{(3)}$. In a linear problem, we normally look at the spectrum over the same frequency range as our source, because other frequencies are zero. In this case, however, we will look from `fcen/2` to `4*fcen`, to be sure that we can see the third-harmonic frequency.

In [ ]:
nfreq = 400
logk = range(-3, 1)
tflux = np.zeros((nfreq, len(logk)))
for i, lk in enumerate(logk):
    freqs, tflux[:, i] = third_harmonic_generation(
        10**lk, nfreq=nfreq, flux_spectrum=True
    )

In [ ]:
fig, ax = plt.subplots()
ax.semilogy(freqs, tflux[:, 0], "bo-", label=r"$\chi^{(3)}$=0.001")
ax.semilogy(freqs, tflux[:, 1], "ro-", label=r"$\chi^{(3)}$=0.01")
ax.semilogy(freqs, tflux[:, 2], "go-", label=r"$\chi^{(3)}$=0.1")
ax.semilogy(freqs, tflux[:, 3], "co-", label=r"$\chi^{(3)}$=1")
ax.set_xlabel("frequency")
ax.set_ylabel("transmitted power (a.u.)")
ax.set_xlim(0.2, 1.2)
ax.set_ylim(1e-15, 1e2)
ax.legend()
ax.grid(True)
plt.show()

For small values of $\chi^{(3)}$, we see a peak from our source at $\omega=\frac{1}{3}$ and another peak 
precisely at the third-harmonic frequency $3\omega=1$. As the $\chi^{(3)}$ gets larger, frequency-mixing 
*within* the peaks causes them to broaden, and finally for $\chi^{(3)}=1$ we start to see a noisy, broad-spectrum 
transmission due to the phenomenon of **modulation instability**. Notice also that at around $10^{-13}$ the 
data looks weird; this is probably due to our finite simulation time, imperfect absorbing boundaries, etcetera. 
We haven't attempted to analyze it in detail for this case.

In the second part of the tutorial, we investigate the dependence of the power at $\omega$ and $3\omega$ as a  function of $\chi^{(3)}$ and the current amplitude. We could, of course, interpolate the flux spectrum above to get the desired frequencies, but it is easier just to add two flux regions to Meep and request exactly the desired frequency components.

In [ ]:
logk = np.arange(-6.0, 0.2, 0.2)
first_order = np.zeros(len(logk))
third_order = np.zeros(len(logk))
for i, lk in enumerate(logk):
    first_order[i], third_order[i] = third_harmonic_generation(
        10**lk, flux_spectrum=False
    )

We divide the transmitted power at all values of $\chi^{(3)}$ by the transmitted power at $\omega$ for the smallest $\chi^{(3)}$ of $10^{-6}$ which is 225.25726603587043. We then plot the fractional transmission at $\omega$ and $3\omega$ as a function of $\chi^{(3)}$ on a log-log scale.

In [ ]:
input_flux = first_order[0]
fig, ax = plt.subplots()
ax.loglog(10**logk, first_order / input_flux, "ro-", label=r"$\omega$")
ax.loglog(10**logk, third_order / input_flux, "bo-", label=r"$3\omega$")
ax.loglog(10**logk, (10**logk) ** 2, "k-", label="quadratic line")
ax.set_xlabel(r"$\chi^{(3)}$")
ax.set_ylabel("transmission / incident power")
ax.legend()
ax.grid(True,'both')
plt.show()

As can be shown from coupled-mode theory or, equivalently, follows from [Fermi's golden rule](https://en.wikipedia.org/wiki/Fermi's_golden_rule), the third-harmonic power must go as the *square* of $\chi^{(3)}$ as long as the nonlinearity is weak (i.e. in the first Born approximation limit, where the $\omega$ source is not depleted significantly). This is precisely what we see on the above graph, where the slope of the black line indicates an exact quadratic dependence, for comparison. Once the nonlinearity becomes strong enough, however, this approximation is no longer valid and the dependence is complicated.

Finally, we note that increasing the current amplitude by a factor of $F$ or the Kerr susceptibility $\chi^{(3)}$ by a factor $F^3$ should generate the *same* third-harmonic power in the *weak* nonlinearity approximation. And indeed, we see:

```
harmonics:, 0.001, 1.0, 225.2091048223644, 0.021498041565849526
```

```
harmonics:, 1e-06, 10.0, 22525.588597389557, 0.021791784143189268
```

which have third-harmonic powers differing by about 1% (last column).